In [48]:
from os import chmod

import pandas as pd
import numpy as np
import networkx as nx
from func.UsefulFunctions import *

In [49]:
#Start with the data for artists

artistDF = pd.read_parquet("Data/Network_artistsWikidata.parquet")


We will be linking artists to their influences, record labels, genres.

In [50]:
artistDF.head()

,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
0,Q3176190,Q639669,Q6581072,musician,female,Q30,NaN,NaN,NaN,Q43343,https://en.wikipedia.org/wiki/Linda_Gail_Lewis,a2fe484f-4f35-4e0e-a954-2f6d5843b009,Q7544232,Smash Records,United States,folk music,Linda Gail Lewis
1,Q8065552,Q183945,Q6581097,record producer,male,Q145,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Zalon,19c43212-18cf-4f02-baff-b794de3cdf02,NaN,NaN,United Kingdom,NaN,Zalon
2,Q191819,Q806349,Q6581097,bandleader,male,Q174193,NaN,NaN,NaN,Q9730,https://en.wikipedia.org/wiki/George_Martin,26fa8b67-6c7f-406c-ad64-a1d070092df2,Q21077,Warner Music Group,United Kingdom of Great Britain and Ireland,classical music,George Martin
3,Q21934026,Q183945,Q6581097,record producer,male,NaN,NaN,NaN,NaN,Q37073,https://en.wikipedia.org/wiki/Andrew_Scheps,ee11737c-d50c-4b04-ae5f-56e1f6373116,NaN,NaN,NaN,pop music,Andrew Scheps
4,Q1005957,Q55960555,Q6581097,recording artist,male,Q28,NaN,NaN,NaN,NaN,NaN,6054baf8-3a47-4c3a-8c51-eb976f69ecfc,NaN,NaN,Hungary,NaN,László Földes


In [51]:
# artistDF["Node_Name"] = artistDF["artist"] + "-" +artistDF["artistLabel"]
# artistDF["Genre_Name"] = artistDF["genre"] + "-" +artistDF["genreLabel"]
# artistDF["recordLabel_Name"] = artistDF["recordLabel"] + "-" +artistDF["recordLabelLabel"]
# artistDF["Influence_Name"] = artistDF["influencedBY"] + "-" +artistDF["influencedBYLabel"]


In [52]:
graphInfluence = addToNetwork4(artistDF, "influencedBY", "artist", "has influenced", "Musical Artist", "Musical artist","influencedBYLabel","artistLabel")


In [53]:
graphRecordLabel = addToNetwork4(artistDF, "recordLabel", "artist", "has/had artist","Record Label", "Musical Artist", "recordLabelLabel", "artistLabel")

In [54]:
graphGenre = addToNetwork4(artistDF, "genre", "artist", "has been participated in by", "Genre", "Musical Artist", "genreLabel", "artistLabel")

In [55]:
#compose all the graphs
firstGraph = nx.compose(graphInfluence, graphRecordLabel)
artistGraph = nx.compose(firstGraph, graphGenre)


In [56]:
#See the nodes in artistGraph
artistGraph.nodes["Q122971758"]

{'EntityType': 'Musical Artist', 'Label': 'Ikkimel'}

In [57]:
nx.write_graphml(artistGraph, "./graphs/artistGraph.graphml")

In [58]:
artistGraph.to_undirected().edges("Q5084390")

EdgeDataView([('Q5084390', 'Q42455'), ('Q5084390', 'Q11975'), ('Q5084390', 'Q21077'), ('Q5084390', 'Q202440'), ('Q5084390', 'Q1998195'), ('Q5084390', 'Q2690060'), ('Q5084390', 'Q37073'), ('Q5084390', 'Q187760'), ('Q5084390', 'Q842324'), ('Q5084390', 'Q188450'), ('Q5084390', 'Q487914'), ('Q5084390', 'Q211756'), ('Q5084390', 'Q1298934'), ('Q5084390', 'Q104695865'), ('Q5084390', 'Q9162904')])

In [59]:
ikkimelEgo = nx.ego_graph(artistGraph, "Q122971758", radius=3, center=True, undirected = True).copy()
print(ikkimelEgo.number_of_nodes())

39


In [60]:
sufjanEgo = nx.ego_graph(artistGraph, "Q319502", radius=3, center=True, undirected = True).copy()
print(sufjanEgo.number_of_nodes())
nx.write_graphml(sufjanEgo, "./graphs/sufjanEgo.graphml")

15549


In [61]:
#Subgraph of bjork, fka twigs, electronic music, hip-hop, arca
twigsEgoGraph = nx.ego_graph(artistGraph, "Q15291033", radius=3, center=True, undirected = True).copy()
#electronicEgoGraph = nx.ego_graph(artistGraph, "Q9778", radius=2, center=True,undirected = True).copy()
#subgraph = nx.compose(electronicEgoGraph, bjorkEgoGraph)
print(len(twigsEgoGraph.nodes()))
nx.write_graphml(twigsEgoGraph, "./graphs/twigsEgoGraph.graphml")
nx.write_graphml(ikkimelEgo, "./graphs/ikkimelEgo.graphml")

591


Make a triphop ego network

In [62]:
#Create a triphop ego network for artists of 1 away
trip_hop_WikidataEgo = nx.ego_graph(artistGraph, "Q205560", radius=1, center=True, undirected = True).copy()
nx.write_graphml(trip_hop_WikidataEgo, "./graphs/trip_hop_WikidataEgo.graphml")

In [63]:
#Create a trip hop ego network for from the data from mb
mbPersonData = pl.read_parquet("./Data/Network_personMBData.parquet")
#explode genre list to get only trip hop
mbPersonData = mbPersonData.explode("genreList", empty_as_null=True)
mbPersonData = mbPersonData.unique()
mbPersonData.head()

id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],str
"""ca12db75-a351-4435-9a17-ff8ee0…","""Milletrax""","""Person""","""Male""",null,[],[],[],"{null,null,false}",null
"""2cb00d05-7dd2-4e73-884a-ea123e…","""Suie Ann Le""","""Person""",null,null,[],[],[],"{null,null,false}",null
"""9ad0ea3c-60c7-4ac6-82cf-7257f1…","""Normandie Wilson""","""Person""",null,"""US""",[],[],[],"{null,null,false}",null
"""9a9bda78-ae2a-4d49-98dc-e843bf…","""Blindskywatcher""","""Person""","""Female""",null,[],[],[],"{null,""1995-11-25"",false}",null
"""fe5bb107-0849-43fa-b934-6b8e15…","""Romuald Pekny""","""Person""","""Male""","""AT""",[],[],[],"{""2007-11-09"",""1920-07-01"",true}",null


In [64]:
#On musicbrainz the genre tripHop is called tripHop
tripHopPeople = mbPersonData.filter(
    pl.col("genreList") == "trip hop"
).select(
    pl.col("id"),
    pl.col("name"),
    pl.col("genreList").alias("genreLabel")
).unique()

tripHopPeople.head()

id,name,genreLabel
str,str,str
"""a72d1627-710b-4ad5-aaff-4bed64…","""Premiere Maxx""","""trip hop"""
"""fd6cfbbd-dce1-4815-9616-334a92…","""Dolàn Xakò""","""trip hop"""
"""3cbe2111-db5a-4bf8-abf9-59a863…","""Miss Melody China""","""trip hop"""
"""526ad2f2-3948-4377-b8fa-370f29…","""Helicopter Girl""","""trip hop"""
"""6080c60a-6e87-44a6-a1e1-e5ff3a…","""Kid Koala""","""trip hop"""


In [65]:
#Lets attempt to find them on wikidata

tripHopPeoplePandas = tripHopPeople.to_pandas()
tripHopPeoplePandas = tripHopPeoplePandas.merge(artistDF[["artist", "musicBrainzID"]], how="left", left_on="id", right_on="musicBrainzID")
tripHopPeoplePandas.drop_duplicates(inplace=True)
tripHopPeoplePandas.head()

,id,name,genreLabel,artist,musicBrainzID
0,a72d1627-710b-4ad5-aaff-4bed64365dde,Premiere Maxx,trip hop,NaN,NaN
1,fd6cfbbd-dce1-4815-9616-334a92b372c3,Dolàn Xakò,trip hop,NaN,NaN
2,3cbe2111-db5a-4bf8-abf9-59a863ceef63,Miss Melody China,trip hop,NaN,NaN
3,526ad2f2-3948-4377-b8fa-370f2976d369,Helicopter Girl,trip hop,Q5705177,526ad2f2-3948-4377-b8fa-370f2976d369
5,6080c60a-6e87-44a6-a1e1-e5ff3a613818,Kid Koala,trip hop,Q1347250,6080c60a-6e87-44a6-a1e1-e5ff3a613818


In [66]:
#Make the musicBrainz ego network for tripHOP

#add all the q-number nodes then add all
artistsWithQNumbers= tripHopPeoplePandas[tripHopPeoplePandas.artist.notna()][["artist","name"]].drop_duplicates()
listOfQnumberTuples = artistsWithQNumbers.itertuples(index=False, name=None)
listOfQnumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist"}) for k, v in listOfQnumberTuples]
#then add all the nodes of individuals without q-numbers
artistsWithOnlyID= tripHopPeoplePandas[tripHopPeoplePandas.artist.isna()][["id","name"]].drop_duplicates()
listOfIdNumberTuples = artistsWithOnlyID.itertuples(index=False, name=None)
listOfIdNumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist"}) for k, v in listOfIdNumberTuples]

#connect them all to triphop

mbTripHopEgo = nx.DiGraph()
#add an vertex for triphop
mbTripHopEgo.add_node("Q205560", Label = "trip hop", EntityType = "Genre")
mbTripHopEgo.add_nodes_from(listOfIdNumberTuples)
mbTripHopEgo.add_nodes_from(listOfQnumberTuples)

for x in mbTripHopEgo.nodes:
    #add edges to triphop
    mbTripHopEgo.add_edge("Q205560", x, Type="has been participated in by (MB)")


#save the graph
nx.write_graphml(mbTripHopEgo, "./graphs/mbTripHopEgo.graphml")




In [67]:
mergedTripHop_Graph = nx.compose(trip_hop_WikidataEgo,mbTripHopEgo)
nx.write_graphml(mergedTripHop_Graph, "./graphs/mergedTripHopEgo.graphml")